# 02 — Data Understanding (CRISP-DM)

Explora a **base bruta canônica** (`dataset_cafeterias_rj.parquet`), validando esquema,
regras de negócio e distribuições. Nada é alterado aqui: apenas diagnóstico.


## Carregamento e esquema

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.data.preparar_dataset import carregar_dados_brutos, validar_esquema

df = carregar_dados_brutos()
print(f"Grão: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

In [ ]:
df.dtypes.to_frame('tipo')

## Relatório de qualidade (fail-closed)

A mesma validação usada no pipeline é aplicada aqui, para que o notebook mostre o
estado real da base.

In [ ]:
problemas = validar_esquema(df)
print("Base íntegra: nenhum problema encontrado." if not problemas else "\n".join(problemas))

## Estatísticas descritivas das variáveis-chave

In [ ]:
colunas = [
    "preco_original", "desconto_pct", "preco_final",
    "custo_producao", "quantidade_vendida_dia", "preco_competidor",
]
df[colunas].describe().round(2)

## Receita por categoria e por unidade

In [ ]:
df["receita_dia"] = df["preco_final"] * df["quantidade_vendida_dia"]
df.groupby("categoria")["receita_dia"].sum().round(2).sort_values(ascending=False)

In [ ]:
df.groupby("loja")["receita_dia"].sum().round(2)

## Sazonalidade: receita mensal

O pico de fim de ano (Q4) é o cenário mais adverso — e será justamente o **teste**
do split temporal na próxima etapa.

In [ ]:
mensal = df.groupby(pd.Grouper(key="data", freq="MS"))["receita_dia"].sum()
ax = mensal.plot(figsize=(10, 4), title="Receita mensal simulada (base bruta)")
ax.set_ylabel("R$")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## Achados

- A base é **coerente por construção**: preço final reconstruível a partir de
  preço original e desconto.
- Há **sazonalidade marcada** (festas de fim de ano) e **diferença entre unidades**.
- Existe uma coluna de **agregado mensal** que precisa ser tratada como vazamento
  na etapa de preparação.
